**Abdelwahid Benslimane**

April 18, 2026.

This notebook complements a report that provides further details on the motivation behind this work and on the broader context. Feel free to contact me if you are unsure where to find it (wahid.besnlimane@gmail.com).

**Importing the Required Libraries**

In the first code block, I gathered all the import statements for the various libraries used throughout the project, so as to resolve from the outset any technical issues related to this aspect, in case the code were to be run in a different environment. 



In [1]:
import cv2
import math
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
from keras.preprocessing import image
import numpy as np
from tensorflow.keras.utils import to_categorical
from skimage.transform import resize
from sklearn.model_selection import train_test_split
from glob import glob
from tqdm import tqdm
import os
from pathlib import Path
import shutil
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import models, transforms
from PIL import Image
import joblib
from sklearn.metrics import classification_report

**Data Integrity Check**

The load_file function iterates over the split files, trainlist01.txt and testlist01.txt, extracting only the video paths (the label is ignored). The duplicates function then verifies the integrity of the splits in two steps: detection of internal duplicates via set(), followed by detection of data shared between the two files via intersection(), ensuring a strict separation of the two datasets before proceeding further. The results show that integrity is guaranteed,  in other words, trainlist01.txt contains 9,537 videos, testlist01.txt contains 3,783 videos (total: 13,320), with no duplicates, and no data common to both. 



In [2]:
def load_file(path):
    videos = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            video_path = line.split()[0]  # on ignore le label
            videos.append(video_path)
    return videos


files = [
    "trainlist01.txt",
    "testlist01.txt"
]


def duplicates(files):
    file_videos = {}

    # Vérifier doublons internes

    for file in files:
        videos = load_file(file)
        unique_videos = set(videos)

        print(f"\nFichier : {file}")
        print("Total lignes :", len(videos))
        print("Unique :", len(unique_videos))

        if len(videos) != len(unique_videos):
            print("Doublons détectés dans le fichier !")
        else:
            print("Aucun doublon interne")

        file_videos[file] = unique_videos


    # Vérifier intersections entre fichiers

    for i in range(len(files)):
        for j in range(i + 1, len(files)):
            f1 = files[i]
            f2 = files[j]

            intersection = file_videos[f1].intersection(file_videos[f2])

            print(f"\nIntersection {f1} / {f2} : {len(intersection)}")

            if len(intersection) > 0:
                print("Vidéos communes détectées !")
            else:
                print("Aucune vidéo commune")
                print(f"\nTotal vidéos:  {len(file_videos[f1]) + len(file_videos[f2])}")

duplicates(files)


Fichier : trainlist01.txt
Total lignes : 9537
Unique : 9537
Aucun doublon interne

Fichier : testlist01.txt
Total lignes : 3783
Unique : 3783
Aucun doublon interne

Intersection trainlist01.txt / testlist01.txt : 0
Aucune vidéo commune

Total vidéos:  13320


**Frame sampling and extraction**

The lists of paths to the train and test videos are loaded via load_file. The extract_frames function then iterates over each video: it opens the file with cv2.VideoCapture, selects 25 evenly distributed frames across the total duration via np.linspace, and saves each frame as a .jpg file in a dedicated folder per video. Uniform sampling ensures consistent temporal coverage regardless of video duration. 


In [3]:
trainfile = load_file("trainlist01.txt")
testfile = load_file("testlist01.txt")

In [4]:
#extraction des frames

def extract_frames(video_list,
                   videos_root,
                   output_root,
                   nb_frames=25):

    os.makedirs(output_root, exist_ok=True)

    for video in video_list:
        
        video_path = videos_root+"/"+video

        if not os.path.exists(video_path):
            print(f"Fichier introuvable : {video_path}")
            continue

        video_name = os.path.splitext(os.path.basename(video_path))[0]

        # 1 dossier par vidéo
        video_dir = os.path.join(output_root, video_name)
        os.makedirs(video_dir, exist_ok=True)

        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            print(f"Impossible d’ouvrir : {video_path}")
            continue

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames <= 0:
            cap.release()
            continue

        frame_indices = np.linspace(0, total_frames - 1, nb_frames, dtype=int)

        for i, idx in enumerate(frame_indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()

            if ret:
                frame_filename = f"{video_name}_frame_{i}.jpg"
                frame_path = os.path.join(video_dir, frame_filename)
                cv2.imwrite(frame_path, frame)

        cap.release()

    print("Extraction terminée.")


In [5]:

extract_frames(trainfile, "UCF-101", "frames_training", nb_frames=25)
extract_frames(testfile, "UCF-101", "frames_test", nb_frames=25)

Extraction terminée.
Extraction terminée.


**Removal of videos with fewer than 25 extracted frames**

The nb_frames_check function iterates over each video folder and detects those not containing exactly 25 frames, a situation that can arise with videos that are too short or corrupted, after which delete_wrong_folders removes these folders via shutil.rmtree. This step ensures that all samples have exactly the same sequence length. Since the number of detected videos is small (42 in the training set, 13 in the test set), and given that no affected class is significantly impacted, this removal will have no effect on training (no bias to be concerned about). 


In [6]:
#vérification du nombre de frames extraites à partir de chaque vidéo

def nb_frames_check(path, nb):

    parent_dir = Path(path)

    wrong_folders = []

    for folder in parent_dir.iterdir():
        if folder.is_dir():
            files = [f for f in folder.iterdir() if f.is_file()]
            if len(files) != nb:
                wrong_folders.append(folder.name)
    print(f"nombre de dossiers ne contenant pas {nb} frames :")
    print(len(wrong_folders))
    print("\n")
    if len(wrong_folders) !=0:
        print(f"dossiers ne contenant pas {nb} frames :")
        print(wrong_folders)

    return wrong_folders



In [7]:
wrong_folders_train = nb_frames_check("frames_training", 25)
print("\n")
wrong_folders_test = nb_frames_check("frames_test", 25)

nombre de dossiers ne contenant pas 25 frames :
42


dossiers ne contenant pas 25 frames :
['v_BaseballPitch_g09_c03', 'v_BaseballPitch_g09_c05', 'v_BaseballPitch_g09_c06', 'v_Basketball_g12_c01', 'v_Basketball_g13_c04', 'v_Basketball_g16_c01', 'v_Basketball_g16_c02', 'v_Basketball_g16_c03', 'v_Basketball_g16_c04', 'v_Basketball_g16_c05', 'v_Basketball_g16_c06', 'v_Basketball_g18_c05', 'v_Diving_g22_c06', 'v_Diving_g23_c06', 'v_GolfSwing_g11_c06', 'v_GolfSwing_g22_c01', 'v_GolfSwing_g22_c02', 'v_GolfSwing_g22_c03', 'v_GolfSwing_g24_c07', 'v_GolfSwing_g25_c07', 'v_HorseRiding_g15_c01', 'v_JumpingJack_g20_c01', 'v_Kayaking_g12_c03', 'v_SoccerJuggling_g14_c03', 'v_SoccerJuggling_g20_c05', 'v_SoccerJuggling_g21_c04', 'v_SoccerJuggling_g24_c05', 'v_SoccerJuggling_g24_c07', 'v_Swing_g25_c04', 'v_TennisSwing_g16_c02', 'v_TennisSwing_g16_c04', 'v_TennisSwing_g23_c01', 'v_TennisSwing_g24_c04', 'v_TrampolineJumping_g14_c01', 'v_VolleyballSpiking_g22_c01', 'v_VolleyballSpiking_g23_c02', 'v_Volley

In [8]:
#suppression des dossiers contenant moins de 25 frames

def delete_wrong_folders(parent_dir_str, wrong_folders):
    parent_dir = Path(parent_dir_str)


    for folder_name in wrong_folders:
        folder_path = parent_dir / folder_name
        if folder_path.exists() and folder_path.is_dir():
            shutil.rmtree(folder_path)

    print("Suppression terminée.")



In [9]:
delete_wrong_folders("frames_training", wrong_folders_train)
delete_wrong_folders("frames_test", wrong_folders_test)

Suppression terminée.
Suppression terminée.


In [10]:
wrong_folders = nb_frames_check("frames_training", 25)
wrong_folders = nb_frames_check("frames_test", 25)

nombre de dossiers ne contenant pas 25 frames :
0


nombre de dossiers ne contenant pas 25 frames :
0




**Feature extraction and creation of usable datasets**

This section creates a VideoDataset for training and one for validation, loading for each video its 25 frames in chronological order (guaranteed by the sort sorted(frames, key=lambda x: int(x.split("_")[-1].split(".")[0])) which orders frames by their final numeric index). Each frame is resized to 224 x 224, converted to a tensor, and normalized before being stacked into a tensor of dimensions (25, 3, 224, 224).

The visual features are then extracted by a pre-trained ResNet-50 (without its final layer), producing for each video a tensor of dimensions (25, 2048), which means a 2048-dimensional feature vector per frame. The entire training set is thus represented by an array of dimensions (number of videos, 25, 2048).

Finally, the training set videos are randomly shuffled via np.random.permutation (with a seed fixed at 42 for reproducibility). This shuffling is necessary because, without it, batches could contain consecutive videos of the same class and bias the learning process. The chronological order of frames within each video is, however, preserved, as the Bi-LSTM depends on it to model the temporal sequence.rieur de chaque vidéo est, quant à lui, préservé, car le Bi-LSTM en dépend pour modéliser la séquence temporelle.

In [11]:
#création du Dataset contenant les liens vers les fichiers vidéos et les labels correspondants
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class VideoDataset(Dataset):
    def __init__(self, frames_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.labels = []

        for video_folder in os.listdir(frames_dir):
            video_path = os.path.join(frames_dir, video_folder)
            if not os.path.isdir(video_path):
                continue
            
            # Extraire la classe depuis le nom du dossier
            cls = video_folder.split('_')[1]
            self.labels.append(cls)

            # Lister les frames
            #frames = sorted([os.path.join(video_path, f) 
            #                 for f in os.listdir(video_path) if f.endswith(".jpg")])
            frames = [os.path.join(video_path, f) 
                             for f in os.listdir(video_path)]
            frames = sorted(frames, key=lambda x: int(x.split("_")[-1].split(".")[0]))
            
            self.samples.append(frames)

        self.label_encoder = LabelEncoder()
        self.label_encoder.fit(self.labels)
        self.labels = self.label_encoder.transform(self.labels)

    def __len__(self):
        return len(self.samples)
        
    #retourne les 25 frames pour chaque vidéo ainsi que le label de la classe
    def __getitem__(self, idx):
        frames_paths = self.samples[idx]
        frames_tensor = []

        for fpath in frames_paths:
            img = Image.open(fpath).convert("RGB")
            if self.transform:
                img = self.transform(img)
            frames_tensor.append(img)

        frames_tensor = torch.stack(frames_tensor)  # #dimensions: (nb frames, 3, 224, 224
        label = self.labels[idx]
        return frames_tensor, label

#normalisation des frames en vue de l'extraction de feature avec Resnet50: https://docs.pytorch.org/vision/0.9/models
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [12]:
dataset_train = VideoDataset("frames_training", transform=transform)
dataloader_train = DataLoader(dataset_train, batch_size=1, shuffle=False)
dataset_test = VideoDataset("frames_test", transform=transform)
dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=False)

In [13]:
#Extraction des features visuelles avec ResNet-50

def extraction_features(dataloader):

    cnn_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    cnn_model = nn.Sequential(*list(cnn_model.children())[:-1])  # retire la dernière couche FC
    cnn_model.eval()
    cnn_model.to(DEVICE)

    # Extraire les features
    all_features = []
    all_labels = []

    with torch.no_grad():
        for frames, label in tqdm(dataloader, desc="Extraction features"):
            frames = frames.squeeze(0).to(DEVICE)  # dimensions: (nb frames, 3, 224, 224)
            features = cnn_model(frames)            # (nb frames, 2048, 1, 1)
            features = features.squeeze(-1).squeeze(-1)  # (nb frames, 2048)
            all_features.append(features.cpu().numpy())
            all_labels.append(label.item())

    all_features = np.array(all_features)  # dimensions: (nb videos, nb frames, 2048)
    all_labels = np.array(all_labels)

    return all_features, all_labels


In [14]:
all_features_train, all_labels_train = extraction_features(dataloader_train)

Extraction features: 100%|██████████| 9495/9495 [3:01:06<00:00,  1.14s/it]  


In [15]:
all_features_test, all_labels_test = extraction_features(dataloader_test)

Extraction features: 100%|██████████| 3770/3770 [1:20:39<00:00,  1.28s/it]


In [16]:
#np.save("all_features_train.npy", all_features_train)
#np.save("all_labels_train.npy", all_labels_train)
#np.save("all_features_test.npy", all_features_test)
#np.save("all_labels_test.npy", all_labels_test)

In [17]:
#all_features_train = np.load("all_features_train.npy")
#all_labels_train = np.load("all_labels_train.npy")
#all_features_test = np.load("all_features_test.npy")
#all_labels_test = np.load("all_labels_test.npy")


In [18]:
#on mélange les vidéos de manière à ne pas à s'assurer que les classes sont réparties de manière totalement aléatoires
np.random.seed(42)
indices = np.random.permutation(len(all_features_train))
all_features_train = all_features_train[indices]
all_labels_train = all_labels_train[indices]


**Dimensionality reduction with PCA**

The data of dimensions (nb videos, 25, 2048) is flattened into a new format (nb videos × 25, 2048) to be compatible with PCA, which reduces the dimensionality from 2048 to 256 components, before being reshaped back to (nb videos, 25, 256). The retained explained variance ratio is computed and displayed, amounting to 93.75%.

In [19]:
#ACP 

n_videos, n_frames, n_features = all_features_train.shape
X_train_flat = all_features_train.reshape(n_videos*n_frames, n_features)
pca = PCA(n_components=256)

X_train_pca = pca.fit_transform(X_train_flat)
X_train_pca = X_train_pca.reshape(n_videos, n_frames, 256)

inertie = np.sum(pca.explained_variance_ratio_) * 100
print(f"Part d'inertie retenue : {inertie:.2f}%")

#on enregistre l'espace réduit produit
#joblib.dump(pca, "pca_features.pkl")

Part d'inertie retenue : 93.75%


['pca_features.pkl']

**Training and evaluation of the Bi-LSTM** 

In this section, I create a Bi-LSTM with the following parameters: input_dim=256 (dimensionality from PCA), hidden_dim=128 (256 after concatenation of both directions), a 30% dropout applied to the output of the last time step, and a fully connected layer 256 -> 101 for classification. 

The PCA-transformed training data is wrapped in a TensorDataset and loaded in batches of 32 samples with shuffling, ensuring videos are mixed at each epoch (in addition to the shuffling applied to the data prior to PCA). The model is trained with a CrossEntropyLoss and the Adam optimizer over 15 epochs.

For evaluation, the test data is first projected into the same PCA space computed previously, then passed all at once to the model in eval() mode, which disables dropout. The predicted class is determined by torch.max on the logits. The model is then saved via state_dict() so that it can be reloaded later if needed, without retraining.
The accuracy achieved on the test data amounts to 75.20%. This result will be analyzed in slightly more detail below.


In [20]:
# LSTM bidirectionnel pour classification avec dropout de 30%

torch.manual_seed(42)
class BiLSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)      
        out = out[:, -1, :]        # on garde la sortie du dernier timestep
        out = self.dropout(out) 
        out = self.fc(out)
        return out

model = BiLSTMClassifier(input_dim=256, hidden_dim=128, num_classes=101, dropout=0.3).to(DEVICE)

In [21]:

#création des tenseurs de données issues de l'ACP 

features_tensor = torch.tensor(X_train_pca, dtype=torch.float32)
labels_tensor = torch.tensor(all_labels_train, dtype=torch.long)

dataset_lstm = TensorDataset(features_tensor, labels_tensor)
dataloader_lstm = DataLoader(dataset_lstm, batch_size=32, shuffle=True)


# Entraînement du modèle bi-LSTM + couche entièrement connectée

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 15

def training(model, dataloader, optimizer, criterion, epochs):
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch, (batch_features, batch_labels) in enumerate(dataloader):

            # envoyer sur device
            batch_features = batch_features.to(DEVICE)
            batch_labels = batch_labels.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1}/{epochs} - Average Loss: {total_loss/len(dataloader):.4f}")

    print("Entraînement terminé !")

training(model, dataloader_lstm, optimizer, criterion, epochs=15)

Epoch 1/15 - Average Loss: 2.4904
Epoch 2/15 - Average Loss: 0.6119
Epoch 3/15 - Average Loss: 0.2704
Epoch 4/15 - Average Loss: 0.1453
Epoch 5/15 - Average Loss: 0.0825
Epoch 6/15 - Average Loss: 0.0473
Epoch 7/15 - Average Loss: 0.0314
Epoch 8/15 - Average Loss: 0.0214
Epoch 9/15 - Average Loss: 0.0148
Epoch 10/15 - Average Loss: 0.0114
Epoch 11/15 - Average Loss: 0.0079
Epoch 12/15 - Average Loss: 0.0063
Epoch 13/15 - Average Loss: 0.0053
Epoch 14/15 - Average Loss: 0.0043
Epoch 15/15 - Average Loss: 0.0058
Entraînement terminé !


In [22]:
#évaluation du modèle
n_videos_test, n_frames, n_features = all_features_test.shape
X_test_flat = all_features_test.reshape(n_videos_test * n_frames, n_features)
X_test_pca = pca.transform(X_test_flat)  
all_features_test_pca = X_test_pca.reshape(n_videos_test, n_frames, 256)

model.eval()

features_test_tensor = torch.tensor(all_features_test_pca, dtype=torch.float32).to(DEVICE)
labels_test_tensor = torch.tensor(all_labels_test, dtype=torch.long).to(DEVICE)

with torch.no_grad():
    outputs_test = model(features_test_tensor)
    _, preds_test = torch.max(outputs_test, 1)
    accuracy_test = (preds_test == labels_test_tensor).float().mean()

print(f"Accuracy test : {accuracy_test.item():.4f}")

Accuracy test : 0.7520


In [23]:
#enregistrement sur disque du modèle 

torch.save(model.state_dict(), "bilstm_inputdim256.pth")

#pour recharger:
#model = BiLSTMClassifier(input_dim=256, hidden_dim=128, num_classes=101, dropout=0.3).to(DEVICE)
#model.load_state_dict(torch.load("bilstm_model.pth", map_location=DEVICE))


**Quick analysis of the results**

In this section, I use the classification_report function, which computes precision, recall, and f1-score for each of the 101 classes, replacing numeric indices with human-readable action names via dataset_train.label_encoder.classes_. It also displays the overall averages: macro avg (unweighted average per class) and weighted avg (weighted by the number of examples per class), as well as the overall accuracy. This allows me to make a few observations.

I notice that among the videos with a high f1-score ( > 0.80) are those involving the playing of a musical instrument (Drumming, PlayingGuitar, PlayingSitar, PlayingFlute, etc.). One can assume that the presence of the musical instrument serves as a strong visual signal that allows the model to recognize the action easily, especially if the instrument is visible from the very first frames. At the same time, these classes, according to Figure 3 of the UCF101 dataset presentation document (see the Dataset description section of this report), tend to have longer videos (> 5 s), for which 25 frames do not densely cover the action. A strong visual signature therefore compensates for a less thorough coverage of the temporal stream by the model.

More generally, activities, particularly sports, that take place in a distinctive, specific, and clearly identifiable environment, and that can be recognized from a single frame or very few frames, tend to achieve high f1-scores. Examples include RockClimbingIndoor, HorseRace, HorseRiding, IceDancing, etc.

Conversely, activities for which the overall visual context does not necessarily provide relevant information, or which do not inherently involve distinctive objects, obtain low f1-scores: JumpRope (0.00), BodyWeightSquats (0.17), etc. Indeed, these activities can take place in very different contexts, as can be observed by watching a few videos from the JumpRope class, for example. Static visual signature therefore appears to be an important discriminating factor for the model.

This analysis is of course brief and remains at a surface level. A more in-depth study taking into account a cross-analysis of the various aspects characterizing the videos would be necessary to compare results and identify patterns leading to poor or strong classification performance in most cases.


In [24]:
#analyse détaillée des résultats

preds_np = preds_test.cpu().numpy()
labels_np = labels_test_tensor.cpu().numpy()

print(classification_report(
    labels_np,
    preds_np,
    target_names=dataset_train.label_encoder.classes_
))

                    precision    recall  f1-score   support

    ApplyEyeMakeup       0.74      0.59      0.66        44
     ApplyLipstick       0.53      0.62      0.57        32
           Archery       0.65      0.76      0.70        41
      BabyCrawling       0.85      1.00      0.92        35
       BalanceBeam       0.64      0.74      0.69        31
      BandMarching       0.85      0.91      0.88        43
     BaseballPitch       0.72      0.85      0.78        40
        Basketball       0.67      0.74      0.70        35
    BasketballDunk       0.77      1.00      0.87        37
        BenchPress       0.94      0.96      0.95        48
            Biking       0.97      0.92      0.95        38
         Billiards       1.00      1.00      1.00        40
       BlowDryHair       0.63      0.71      0.67        38
    BlowingCandles       0.69      0.94      0.79        33
  BodyWeightSquats       0.17      0.13      0.15        30
           Bowling       0.90      0.88

**Training a Bi-LSTM on "raw" data (without PCA)**

A Bi-LSTM is trained this time without PCA, using the "raw" features from ResNet-50 as input (input_dim=2048 instead of 256), with all other hyperparameters remaining identical (hidden_dim=128, dropout=0.3, 15 epochs, batch_size=32). The accuracy obtained is 72.39%, slightly lower than that achieved with PCA, suggesting that in this case dimensionality reduction did not degrade the discriminative information and in fact helped the model generalize better, while also enabling faster training and convergence at a lower computational cost. 


In [25]:
#entraînement d'un modèle avec en entrée des données de dimension (25, 2048)

model = BiLSTMClassifier(input_dim=2048, hidden_dim=128, num_classes=101, dropout=0.3).to(DEVICE)

features_tensor = torch.tensor(all_features_train, dtype=torch.float32)
labels_tensor = torch.tensor(all_labels_train, dtype=torch.long)
features_tensor = torch.tensor(all_features_train, dtype=torch.float32)
labels_tensor = torch.tensor(all_labels_train, dtype=torch.long)

dataset_lstm = TensorDataset(features_tensor, labels_tensor)
dataloader_lstm = DataLoader(dataset_lstm, batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

training(model, dataloader_lstm, optimizer, criterion, epochs=15)



Epoch 1/15 - Average Loss: 3.2298
Epoch 2/15 - Average Loss: 1.5913
Epoch 3/15 - Average Loss: 1.0350
Epoch 4/15 - Average Loss: 0.7485
Epoch 5/15 - Average Loss: 0.5775
Epoch 6/15 - Average Loss: 0.4570
Epoch 7/15 - Average Loss: 0.3788
Epoch 8/15 - Average Loss: 0.3002
Epoch 9/15 - Average Loss: 0.2619
Epoch 10/15 - Average Loss: 0.2157
Epoch 11/15 - Average Loss: 0.1832
Epoch 12/15 - Average Loss: 0.1645
Epoch 13/15 - Average Loss: 0.1480
Epoch 14/15 - Average Loss: 0.1250
Epoch 15/15 - Average Loss: 0.1261
Entraînement terminé !


In [26]:
#évaluation du nouveau modèle précédent 

model.eval()

features_test_tensor = torch.tensor(all_features_test, dtype=torch.float32).to(DEVICE)
labels_test_tensor = torch.tensor(all_labels_test, dtype=torch.long).to(DEVICE)

with torch.no_grad():
    outputs_test = model(features_test_tensor)
    _, preds_test = torch.max(outputs_test, 1)
    accuracy_test = (preds_test == labels_test_tensor).float().mean()

print(f"Accuracy test : {accuracy_test.item():.4f}")

Accuracy test : 0.7239


**Training a Bi-LSTM on "raw" data with increased hidden_dim**

This time, a model with hidden_dim doubled to 256 (512 after concatenation of both directions) is trained again on raw ResNet-50 features to increase the model's capacity, with all other parameters remaining identical. The accuracy drops to 69.84%, indicating that the increased capacity has reduced generalization ability. With only 15 epochs and no data augmentation, the model is unable to make good use of the additional parameters. 


In [27]:
#maintenant, augmentons le paramètre hidden_dim pour voir si les résultats sont meilleurs

model = BiLSTMClassifier(input_dim=2048, hidden_dim=256, num_classes=101, dropout=0.3).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

training(model, dataloader_lstm, optimizer, criterion, epochs=15)


Epoch 1/15 - Average Loss: 2.8421
Epoch 2/15 - Average Loss: 1.2184
Epoch 3/15 - Average Loss: 0.7467
Epoch 4/15 - Average Loss: 0.5297
Epoch 5/15 - Average Loss: 0.3860
Epoch 6/15 - Average Loss: 0.2893
Epoch 7/15 - Average Loss: 0.2300
Epoch 8/15 - Average Loss: 0.1794
Epoch 9/15 - Average Loss: 0.1379
Epoch 10/15 - Average Loss: 0.1190
Epoch 11/15 - Average Loss: 0.0923
Epoch 12/15 - Average Loss: 0.0737
Epoch 13/15 - Average Loss: 0.0711
Epoch 14/15 - Average Loss: 0.0788
Epoch 15/15 - Average Loss: 0.0676
Entraînement terminé !


In [28]:
with torch.no_grad():
    outputs_test = model(features_test_tensor)
    _, preds_test = torch.max(outputs_test, 1)
    accuracy_test = (preds_test == labels_test_tensor).float().mean()

print(f"Accuracy test : {accuracy_test.item():.4f}")

Accuracy test : 0.6984
